# Artificial Neural Network (ANN) — Complete Practical Notebook

## Goal
This notebook is designed for **learning, practice, and revision** of Artificial Neural Networks from the basics to a complete binary-classification project.

You will learn:
1. What an ANN is and why it is used
2. Neurons, weights, bias, activation functions, layers and architecture
3. Forward propagation
4. Loss / cost function
5. Backpropagation and gradient descent
6. Optimizers
7. Epoch, batch, iteration
8. Building an ANN with Keras
9. Data preprocessing and feature scaling
10. Training, validation and testing
11. Evaluation metrics and confusion matrix
12. Overfitting and regularization
13. Hyperparameter tuning concepts
14. Saving and loading a trained model
15. Making predictions on new data

> **Note:** Run the notebook from top to bottom. Each section contains explanations followed by practical code.

## 1. Environment Check

We are using the TensorFlow environment (`tf_env`). This cell verifies the important libraries before starting.

In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import tensorflow as tf

print("Python:", sys.version.split()[0])
print("Python executable:", sys.executable)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("Matplotlib:", plt.matplotlib.__version__)
print("Scikit-learn:", sklearn.__version__)
print("TensorFlow:", tf.__version__)

## 2. What is an Artificial Neural Network?

An **Artificial Neural Network (ANN)** is a machine-learning model inspired by the way biological neurons process information.

A simple ANN contains:

**Input → Hidden Layer(s) → Output**

### Important terms

- **Input:** Features given to the network.
- **Neuron / Node:** A small computational unit.
- **Weight:** Controls how strongly an input influences a neuron.
- **Bias:** Helps shift the neuron's output.
- **Weighted sum:** Inputs are multiplied by weights and added with bias.
- **Activation function:** Converts the weighted sum into the neuron's output.
- **Hidden layer:** Learns intermediate patterns.
- **Output layer:** Produces the final prediction.

A single neuron performs approximately:

`z = w1*x1 + w2*x2 + ... + b`

Then:

`output = activation(z)`

## 3. Neuron Example

Suppose a neuron receives two inputs:

- x1 = 2
- x2 = 3
- w1 = 0.5
- w2 = 0.2
- bias = 0.1

The neuron first calculates the weighted sum and then applies an activation function.

In [ ]:
x1, x2 = 2, 3
w1, w2 = 0.5, 0.2
b = 0.1

z = x1*w1 + x2*w2 + b
print("Weighted sum (z):", z)

# Sigmoid activation
sigmoid = 1 / (1 + np.exp(-z))
print("Sigmoid output:", sigmoid)

## 4. Activation Functions

Activation functions introduce **non-linearity**, allowing neural networks to learn complex relationships.

### Common activation functions

**ReLU**

`ReLU(x) = max(0, x)`

Commonly used in hidden layers.

**Sigmoid**

Produces values between 0 and 1. Common for binary classification output.

**Tanh**

Produces values between -1 and 1.

**Softmax**

Converts multiple output scores into probabilities whose sum is 1. Common for multi-class classification.

### Practical visualization

In [ ]:
x = np.linspace(-6, 6, 400)

relu = np.maximum(0, x)
sigmoid = 1 / (1 + np.exp(-x))
tanh = np.tanh(x)

plt.figure(figsize=(12, 4))

plt.subplot(1, 3, 1)
plt.plot(x, relu)
plt.title("ReLU")
plt.grid(True)

plt.subplot(1, 3, 2)
plt.plot(x, sigmoid)
plt.title("Sigmoid")
plt.grid(True)

plt.subplot(1, 3, 3)
plt.plot(x, tanh)
plt.title("Tanh")
plt.grid(True)

plt.tight_layout()
plt.show()

## 5. Forward Propagation

**Forward propagation** means data moves from the input layer toward the output layer.

Flow:

**Input → Weighted Sum → Activation → Next Layer → Output**

At every layer, the network calculates:

`z = XW + b`

and then:

`a = activation(z)`

The final activation gives the prediction.

## 6. Loss Function

The model needs a way to measure how wrong its predictions are. This measurement is called the **loss**.

For binary classification, a common loss is **Binary Cross-Entropy**.

A simplified idea is:

- correct confident prediction → small loss
- incorrect confident prediction → large loss

During training, the network tries to minimize the loss.

## 7. Backpropagation

**Backpropagation** calculates how much each weight contributed to the error.

Basic training flow:

1. Forward propagation
2. Calculate prediction
3. Calculate loss
4. Backpropagate the error
5. Calculate gradients
6. Update weights
7. Repeat

The update is conceptually:

`new_weight = old_weight - learning_rate × gradient`

The **gradient** tells the optimizer which direction can reduce the loss.

## 8. Gradient Descent and Optimizers

**Gradient Descent** is an optimization method used to reduce the loss.

Common optimizers in neural networks:

- SGD — Stochastic Gradient Descent
- Adam — Adaptive Moment Estimation
- RMSprop
- Adagrad

**Adam** is a popular default choice because it adapts the learning rate for different parameters.

## 9. Epoch, Batch and Iteration

### Epoch
One complete pass through the entire training dataset.

### Batch
A smaller group of training samples processed together.

### Iteration
One weight-update step for one batch.

Example:

- 1,000 training samples
- batch size = 100

Then one epoch contains:

`1000 / 100 = 10 iterations`

If training runs for 20 epochs:

`20 × 10 = 200 weight-update iterations`

# 10. Complete ANN Project — Binary Classification

We will create a realistic binary-classification workflow.

Dataset: **Breast Cancer Wisconsin dataset** available through scikit-learn.

Target:
- 0 = malignant
- 1 = benign

This is a learning dataset. The purpose here is to practice the ANN workflow, not to build a clinical diagnostic system.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()

X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

print("X shape:", X.shape)
print("y shape:", y.shape)

display(X.head())
display(y.head())

## 11. Understand the Dataset

In [ ]:
print("Number of samples:", X.shape[0])
print("Number of features:", X.shape[1])
print("\nTarget counts:")
print(y.value_counts())

print("\nMissing values:")
print(X.isnull().sum().sum())

print("\nData types:")
display(X.dtypes.value_counts())

## 12. Basic EDA

Before training a model, inspect the data.

Questions:
- How many observations are present?
- How many features?
- Are there missing values?
- Is the target balanced?
- What are the feature distributions?

In [ ]:
y.value_counts().sort_index().plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.xticks(rotation=0)
plt.show()

In [ ]:
X.describe().T.head(15)

## 13. Train-Test Split

We should not train and evaluate on the same data.

- **Training data:** used to learn parameters.
- **Validation data:** used during model development.
- **Test data:** final unseen evaluation.

We first create training and test sets using a stratified split so the class proportions are approximately preserved.

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])

## 14. Feature Scaling

Neural networks generally train better when numerical features are on comparable scales.

A common technique is **StandardScaler**:

`z = (x - mean) / standard_deviation`

Important rule:

**Fit the scaler only on training data.**

Then use the same fitted scaler to transform training and test data.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print("Training mean of first feature:", X_train_scaled[:, 0].mean())
print("Training std of first feature:", X_train_scaled[:, 0].std())

## 15. Build the ANN with Keras

We will use:

- Input layer: 30 features
- Hidden layer 1: 32 neurons + ReLU
- Hidden layer 2: 16 neurons + ReLU
- Output layer: 1 neuron + Sigmoid

Why sigmoid at the output?

Because this is a binary classification problem and sigmoid gives a value between 0 and 1 that can be interpreted as a probability-like score.

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers

model = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model.summary()

## 16. Compile the Model

Compilation defines:

### Optimizer
How weights are updated. We use **Adam**.

### Loss
For binary classification, use **binary_crossentropy**.

### Metrics
We track accuracy during training.

Conceptually:

`model.compile(optimizer, loss, metrics)`

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

print("Model compiled successfully.")

## 17. Train the ANN

We use:

- epochs = 50
- batch_size = 32
- validation_split = 0.20

The validation data is kept separate from the data used to update weights.

The returned `history` object stores training and validation loss/accuracy for each epoch.

In [ ]:
history = model.fit(
    X_train_scaled,
    y_train,
    epochs=50,
    batch_size=32,
    validation_split=0.20,
    verbose=1
)

## 18. Understand the Training Output

During training you will see values such as:

- `loss`
- `accuracy`
- `val_loss`
- `val_accuracy`

### What they mean

**loss:** training error

**accuracy:** percentage of correct training predictions

**val_loss:** error on validation data

**val_accuracy:** accuracy on validation data

Do not judge a model only by training accuracy. Compare training and validation behavior to identify overfitting.

## 19. Plot Training and Validation Loss

In [ ]:
history_df = pd.DataFrame(history.history)

plt.figure(figsize=(8, 5))
plt.plot(history_df["loss"], label="Training Loss")
plt.plot(history_df["val_loss"], label="Validation Loss")
plt.title("Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend()
plt.grid(True)
plt.show()

## 20. Plot Training and Validation Accuracy

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(history_df["accuracy"], label="Training Accuracy")
plt.plot(history_df["val_accuracy"], label="Validation Accuracy")
plt.title("Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.legend()
plt.grid(True)
plt.show()

## 21. Evaluate on Completely Unseen Test Data

The test set was not used for fitting the network.

This gives a more honest estimate of generalization performance.

In [ ]:
test_loss, test_accuracy = model.evaluate(
    X_test_scaled,
    y_test,
    verbose=0
)

print("Test Loss:", test_loss)
print("Test Accuracy:", test_accuracy)

## 22. Make Predictions

`model.predict()` produces probability-like outputs because the output layer uses sigmoid.

For binary classification we can convert probabilities to classes using a threshold of 0.5:

- probability >= 0.5 → class 1
- probability < 0.5 → class 0

In [ ]:
y_prob = model.predict(X_test_scaled, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("First 10 probabilities:")
print(y_prob[:10])

print("\nFirst 10 predicted classes:")
print(y_pred[:10])

print("\nFirst 10 actual classes:")
print(y_test.to_numpy()[:10])

## 23. Classification Metrics

Accuracy alone can be misleading, especially with imbalanced classes.

We will calculate:

- Accuracy
- Precision
- Recall
- F1-score
- Confusion matrix

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)

print("Accuracy :", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall   :", recall_score(y_test, y_pred))
print("F1 Score :", f1_score(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

## 24. Confusion Matrix

For binary classification:

- **True Positive (TP):** predicted positive and actually positive
- **True Negative (TN):** predicted negative and actually negative
- **False Positive (FP):** predicted positive but actually negative
- **False Negative (FN):** predicted negative but actually positive

A confusion matrix helps us understand the types of mistakes.

In [ ]:
cm = confusion_matrix(y_test, y_pred)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=data.target_names
)

disp.plot()
plt.title("ANN Confusion Matrix")
plt.show()

## 25. Overfitting

**Overfitting** happens when a model learns the training data too closely and performs worse on unseen data.

Typical signs:

- Training accuracy keeps increasing
- Validation accuracy stops improving or decreases
- Training loss keeps decreasing
- Validation loss starts increasing

### Ways to reduce overfitting

1. More training data
2. Simpler architecture
3. Dropout
4. L2 regularization
5. Early stopping
6. Data augmentation for suitable domains
7. Careful hyperparameter tuning

## 26. Early Stopping

Early stopping monitors validation performance.

If validation loss stops improving for several epochs, training can stop automatically and restore the best weights.

This can prevent unnecessary training and reduce overfitting.

In [ ]:
early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True
)

model_es = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dense(16, activation="relu"),
    layers.Dense(1, activation="sigmoid")
])

model_es.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

history_es = model_es.fit(
    X_train_scaled,
    y_train,
    epochs=100,
    batch_size=32,
    validation_split=0.20,
    callbacks=[early_stopping],
    verbose=1
)

print("Training stopped after", len(history_es.history["loss"]), "epochs.")

## 27. Dropout

**Dropout** randomly turns off a fraction of neurons during training.

Example:

`Dropout(0.30)`

means approximately 30% of the selected activations are dropped during training.

This is a regularization technique that can help reduce overfitting.

In [ ]:
model_dropout = keras.Sequential([
    layers.Input(shape=(X_train_scaled.shape[1],)),
    layers.Dense(32, activation="relu"),
    layers.Dropout(0.30),
    layers.Dense(16, activation="relu"),
    layers.Dropout(0.20),
    layers.Dense(1, activation="sigmoid")
])

model_dropout.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model_dropout.summary()

## 28. Model Architecture Practice

Try changing one thing at a time and observe the effect.

Examples:

### Experiment A
Change hidden neurons:
- 64 → 32

### Experiment B
Add another hidden layer:
- 32 → 16 → 8

### Experiment C
Change learning rate:
- 0.001
- 0.0005
- 0.01

### Experiment D
Change batch size:
- 16
- 32
- 64

### Experiment E
Change optimizer:
- Adam
- SGD
- RMSprop

Do not change many parameters at once if your goal is to understand their individual effects.

## 29. Optimizer Practice

Compare a few optimizers on separate models.

### Adam
Usually a strong starting point.

### SGD
Simple gradient-based optimizer. Momentum can improve its behavior.

### RMSprop
Adapts learning rates based on recent gradient magnitudes.

The best optimizer depends on the problem; there is no universal winner.

In [ ]:
def build_model(optimizer):
    m = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(32, activation="relu"),
        layers.Dense(16, activation="relu"),
        layers.Dense(1, activation="sigmoid")
    ])
    m.compile(
        optimizer=optimizer,
        loss="binary_crossentropy",
        metrics=["accuracy"]
    )
    return m

optimizers = {
    "Adam": keras.optimizers.Adam(learning_rate=0.001),
    "SGD": keras.optimizers.SGD(learning_rate=0.01, momentum=0.9),
    "RMSprop": keras.optimizers.RMSprop(learning_rate=0.001)
}

results = []

for name, optimizer in optimizers.items():
    m = build_model(optimizer)
    h = m.fit(
        X_train_scaled,
        y_train,
        epochs=20,
        batch_size=32,
        validation_split=0.20,
        verbose=0
    )
    loss, acc = m.evaluate(X_test_scaled, y_test, verbose=0)
    results.append([name, loss, acc])

optimizer_results = pd.DataFrame(
    results,
    columns=["Optimizer", "Test Loss", "Test Accuracy"]
)

optimizer_results

## 30. Save the Trained Model

Saving the model allows you to use it later without retraining from scratch.

Keras supports the `.keras` format.

We also save the scaler because preprocessing applied during training must be applied in exactly the same way to future input data.

In [ ]:
import joblib

model_path = "ann_breast_cancer.keras"
scaler_path = "ann_scaler.joblib"

model_es.save(model_path)
joblib.dump(scaler, scaler_path)

print("Saved model:", model_path)
print("Saved scaler:", scaler_path)

## 31. Load the Model and Scaler

In [ ]:
loaded_model = keras.models.load_model(model_path)
loaded_scaler = joblib.load(scaler_path)

loaded_loss, loaded_accuracy = loaded_model.evaluate(
    loaded_scaler.transform(X_test),
    y_test,
    verbose=0
)

print("Loaded model test accuracy:", loaded_accuracy)

## 32. Predict a New Sample

For a new observation:

1. Put the features into the correct order.
2. Apply the **already fitted scaler**.
3. Pass the scaled values to the trained model.
4. Convert the probability into a class if needed.

Never fit a new scaler on a single prediction sample.

In [ ]:
sample = X_test.iloc[[0]]

sample_scaled = loaded_scaler.transform(sample)
sample_probability = loaded_model.predict(sample_scaled, verbose=0)[0][0]
sample_class = int(sample_probability >= 0.5)

print("Predicted probability:", sample_probability)
print("Predicted class:", sample_class)
print("Actual class:", int(y_test.iloc[0]))

# 33. Complete ANN Workflow — Remember This

```text
Raw Dataset
     ↓
Understand Dataset
     ↓
EDA
     ↓
Clean / Prepare Data
     ↓
Train-Test Split
     ↓
Feature Scaling
     ↓
Build ANN
     ↓
Compile
     ↓
Forward Propagation
     ↓
Loss Calculation
     ↓
Backpropagation
     ↓
Optimizer Updates Weights
     ↓
Repeat for Epochs
     ↓
Validation
     ↓
Test Evaluation
     ↓
Predictions
     ↓
Save Model + Preprocessor
```

This is the practical ANN pipeline you should remember.

# 34. Important Concepts — Quick Revision

### Neural Network
A collection of connected neurons that learns patterns from data.

### Weight
A trainable value controlling the influence of an input.

### Bias
A trainable value that shifts the weighted sum.

### Activation Function
Introduces non-linearity.

### Forward Propagation
Moves input data through the network to produce a prediction.

### Loss Function
Measures prediction error.

### Backpropagation
Computes gradients of the loss with respect to trainable parameters.

### Gradient
Indicates how a parameter affects the loss.

### Optimizer
Uses gradients to update model parameters.

### Learning Rate
Controls the size of each update.

### Epoch
One complete pass through training data.

### Batch
Subset of training samples processed in one step.

### Iteration
One parameter-update step.

### Overfitting
Excellent training performance but weaker generalization.

### Regularization
Techniques used to reduce overfitting, such as dropout and L2 regularization.

# 35. Practice Exercises

## Beginner
1. Change the number of neurons in the first hidden layer from 32 to 64.
2. Change ReLU to tanh in the hidden layers.
3. Change batch size from 32 to 16.
4. Train for 10, 30 and 50 epochs and compare the curves.
5. Print the model summary and count the trainable parameters.

## Intermediate
6. Add another hidden layer.
7. Add dropout and compare validation performance.
8. Compare Adam, SGD and RMSprop.
9. Change the learning rate and observe the training curve.
10. Compare accuracy, precision, recall and F1-score.

## Advanced
11. Use a custom threshold such as 0.4 instead of 0.5 and compare recall.
12. Build an ANN using a different binary-classification dataset.
13. Add L2 regularization.
14. Use EarlyStopping and compare it with fixed 100-epoch training.
15. Save the model and scaler, restart the notebook, load them, and predict a sample.
16. Create your own CSV classification dataset and run the entire workflow.
17. Compare ANN performance with Logistic Regression, Decision Tree and Random Forest.
18. Experiment with class weights if the dataset is imbalanced.

# 36. Common Interview Questions

### Q1. What is an ANN?
An ANN is a machine-learning model made of interconnected neurons that learns relationships between input features and target outputs.

### Q2. Why do we need activation functions?
Without suitable non-linear activation functions, stacking linear layers would still behave like a linear transformation. Activation functions allow the network to learn complex non-linear patterns.

### Q3. What is a weight?
A trainable parameter that controls the contribution of an input or previous-layer activation.

### Q4. What is bias?
A trainable parameter added to the weighted sum to shift the activation.

### Q5. What is forward propagation?
The process of passing input data through the layers to calculate an output.

### Q6. What is backpropagation?
The process of calculating gradients of the loss with respect to model parameters using the chain rule.

### Q7. What is an epoch?
One complete pass through the training dataset.

### Q8. What is batch size?
The number of samples processed before one parameter update.

### Q9. Why is scaling useful for ANN?
It can make optimization more stable and prevent features with large numerical ranges from dominating the optimization process.

### Q10. Why use sigmoid for binary classification?
It maps the output to a value between 0 and 1, making it suitable for binary probability-like predictions.

### Q11. Why use binary cross-entropy?
It is a standard loss function for binary classification and measures disagreement between binary targets and predicted probabilities.

### Q12. What is an optimizer?
An algorithm that updates trainable parameters using gradient information to minimize the loss.

### Q13. What is learning rate?
A hyperparameter controlling the size of parameter updates.

### Q14. What is overfitting?
When a model performs very well on training data but does not generalize well to unseen data.

### Q15. How can overfitting be reduced?
Use early stopping, dropout, regularization, simpler architectures, more data, or suitable data augmentation.

### Q16. What is the difference between parameters and hyperparameters?
Weights and biases are learned parameters. Learning rate, number of layers, neurons and batch size are examples of hyperparameters.

### Q17. Why should the scaler be fitted only on training data?
Fitting it using test data causes data leakage because information from the test set influences preprocessing.

### Q18. Why should we keep the scaler when saving a model?
Future data must receive the same preprocessing transformation used during training.

### Q19. What does `model.fit()` do?
It trains the model by repeatedly processing batches, calculating loss, computing gradients and updating parameters.

### Q20. What does `model.evaluate()` do?
It evaluates the model on supplied data without performing training updates.

# 37. Final Practice Challenge — Build It Yourself

After understanding the notebook, create a new notebook and reproduce the project **without copying the solution cells**.

Your checklist:

1. Load a binary classification dataset.
2. Inspect rows, columns and target.
3. Check missing values.
4. Perform basic EDA.
5. Split train/test data.
6. Scale the features.
7. Build an ANN.
8. Compile with an appropriate loss and optimizer.
9. Train using validation data.
10. Plot loss.
11. Plot accuracy.
12. Evaluate test data.
13. Generate predictions.
14. Create a confusion matrix.
15. Print classification metrics.
16. Add EarlyStopping.
17. Experiment with dropout.
18. Compare optimizers.
19. Save the model.
20. Save the scaler.
21. Load both again.
22. Make a prediction on unseen data.

**If you can complete this independently, you have practiced the complete basic-to-intermediate ANN workflow.**